# Distributed Smart-Grid Demand & Tariff Analytics
## Practical Lab 1: Pseudo-Distributed Hadoop HDFS Setup & The 7 Core PySpark RDD Operations

---

### 👥 Course Project Team (4th-Year Big Data Analytics)
- **Ahmed Saad**
- **Mohamed Abu-elhajjaj**
- **Shrief Abdelmoati**
- **Moustafa Mahmoud**
- **Sameh Esam**

---

### 📖 English Overview
This notebook demonstrates the end-to-end execution of **Spec 01 (Phase 1)** on Google Colab:
1. Automated provisioning of **Apache Hadoop 3.4.1** in single-node pseudo-distributed mode (`hdfs://localhost:9000`, `dfs.replication=1`).
2. Initialization of the **Three-Layer HDFS Storage Hierarchy** (`/raw`, `/cleaned`, `/analytics`).
3. Ingestion of raw **Smart Meters in London** CSV data into `/raw` (immutable source tier).
4. Execution of the **7 core low-level PySpark RDD transformations** (`filter`, `map`, `flatMap`, `reduce`, `groupByKey`, `reduceByKey`, `join`).
5. Comparative evaluation of Dynamic Time-of-Use (`ToU`) vs. Standard (`Std`) tariffs with strict Driver memory protection (<= 100 rows collected).

### 📖 نظرة عامة باللغة العربية
يوضح هذا الدفتر التنفيذ الكامل للمرحلة الأولى من المشروع على Google Colab:
1. إعداد وتثبيت نظام **Apache Hadoop 3.4.1** بنمط العقدة الواحدة الموزعة (`hdfs://localhost:9000` مع عامل تكرار 1).
2. إنشاء هيكل التخزين ثلاثي الطبقات على HDFS (`/raw`, `/cleaned`, `/analytics`).
3. تحميل ملفات قراءات العدادات الذكية في لندن إلى مجلد `/raw` دون التعديل عليها.
4. تطبيق العمليات السبع الأساسية للـ RDDs في PySpark:
   (`filter`, `map`, `flatMap`, `reduce`, `groupByKey`, `reduceByKey`, `join`).
5. مقارنة إجمالي ومتوسط استهلاك الكهرباء لتعريفة الاستخدام المتغير (ToU) مقابل التعريفة الثابتة (Std) مع حماية ذاكرة الـ Driver.


## 1. System Environment & Installing Hadoop 3.4.1
## 1. تثبيت بيئة العمل وحزمة Apache Hadoop 3.4.1


In [ ]:
# Install OpenJDK 11 and OpenSSH Server for pseudo-distributed Hadoop
!apt-get update -qq > /dev/null
!apt-get install -y -qq openjdk-11-jdk-headless openssh-server > /dev/null

# Download Apache Hadoop 3.4.1
!wget -q -nc https://archive.apache.org/dist/hadoop/common/hadoop-3.4.1/hadoop-3.4.1.tar.gz
!tar -xzf hadoop-3.4.1.tar.gz -C /usr/local/
!mv /usr/local/hadoop-3.4.1 /usr/local/hadoop

# Set environment variables
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["HADOOP_HOME"] = "/usr/local/hadoop"
os.environ["HADOOP_CONF_DIR"] = "/usr/local/hadoop/etc/hadoop"
os.environ["PATH"] += ":/usr/local/hadoop/bin:/usr/local/hadoop/sbin"

print("Hadoop binary installed successfully at /usr/local/hadoop")


## 2. Hadoop XML Configuration & Passwordless SSH
## 2. إعداد ملفات التكوين وإنشاء اتصال SSH بدون كلمة سر


In [ ]:
# Setup passwordless SSH for localhost
!service ssh start > /dev/null
!ssh-keygen -t rsa -P "" -f ~/.ssh/id_rsa -q
!cat ~/.ssh/id_rsa.pub >> ~/.ssh/authorized_keys
!chmod 0600 ~/.ssh/authorized_keys
!ssh-keyscan -H localhost >> ~/.ssh/known_hosts 2>/dev/null

# Configure hadoop-env.sh
with open("/usr/local/hadoop/etc/hadoop/hadoop-env.sh", "a") as f:
    f.write("\nexport JAVA_HOME=/usr/lib/jvm/java-11-openjdk-amd64\n")

# 1. core-site.xml (Constitutional Principle I: Default FS hdfs://localhost:9000)
core_site = '''<?xml version="1.0" encoding="UTF-8"?>
<?xml-stylesheet type="text/xsl" href="configuration.xsl"?>
<configuration>
    <property>
        <name>fs.defaultFS</name>
        <value>hdfs://localhost:9000</value>
    </property>
</configuration>'''
with open("/usr/local/hadoop/etc/hadoop/core-site.xml", "w") as f:
    f.write(core_site)

# 2. hdfs-site.xml (dfs.replication=1 for single-node cluster)
hdfs_site = '''<?xml version="1.0" encoding="UTF-8"?>
<?xml-stylesheet type="text/xsl" href="configuration.xsl"?>
<configuration>
    <property>
        <name>dfs.replication</name>
        <value>1</value>
    </property>
    <property>
        <name>dfs.namenode.name.dir</name>
        <value>file:///tmp/hadoop-root/dfs/name</value>
    </property>
    <property>
        <name>dfs.datanode.data.dir</name>
        <value>file:///tmp/hadoop-root/dfs/data</value>
    </property>
</configuration>'''
with open("/usr/local/hadoop/etc/hadoop/hdfs-site.xml", "w") as f:
    f.write(hdfs_site)

print("core-site.xml and hdfs-site.xml configured successfully!")


## 3. Formatting NameNode & Starting HDFS Daemons
## 3. تهيئة الـ NameNode وتشغيل عمليات HDFS


In [ ]:
# Format NameNode filesystem
!hdfs namenode -format -force > /dev/null

# Start HDFS daemons (NameNode, DataNode, SecondaryNameNode)
!/usr/local/hadoop/sbin/start-dfs.sh

# Check active Java processes using JPS
!jps


## 4. Initializing the Three-Layer HDFS Storage Layout
## 4. إنشاء الطبقات الثلاث لتخزين البيانات على HDFS (`/raw`, `/cleaned`, `/analytics`)

> **Constitutional Principle I**: Strict separation between immutable raw inputs, cleaned Parquet tables, and analytical summary marts.


In [ ]:
# Create the 3 isolated directory layers
!hdfs dfs -mkdir -p /raw /cleaned /analytics
!hdfs dfs -ls /


## 5. Ingesting Raw Smart Meter Data into `/raw`
## 5. تحميل بيانات العدادات الذكية إلى طبقة `/raw` غير القابلة للتعديل


In [ ]:
!mkdir -p /tmp/sample_data
sample_meters = '''LCLid,day,energy(kWh/hh)
MAC000001,2013-01-01 00:30:00.0000000,0.250
MAC000001,2013-01-01 01:00:00.0000000,0.310
MAC000001,2013-01-01 01:30:00.0000000,0.180
MAC000001,2013-01-01 02:00:00.0000000,0.210
MAC000002,2013-01-01 00:30:00.0000000,0.850
MAC000002,2013-01-01 01:00:00.0000000,0.920
MAC000002,2013-01-01 01:30:00.0000000,0.780
MAC000002,2013-01-01 02:00:00.0000000,0.810
MAC000003,2013-01-01 00:30:00.0000000,1.250
MAC000003,2013-01-01 01:00:00.0000000,1.350
MAC000003,2013-01-01 01:30:00.0000000,1.100
MAC000003,2013-01-01 02:00:00.0000000,1.550
MAC000001,2013-01-01 02:30:00.0000000,Null
MAC000002,2013-01-01 02:30:00.0000000,-0.150
MAC000003,corrupted_line_without_columns
'''
sample_households = '''LCLid,stdorToU,Acorn,Acorn_grouped,file
MAC000001,Std,ACORN-A,Affluent,block_0
MAC000002,ToU,ACORN-B,Affluent,block_0
MAC000003,Std,ACORN-E,Comfortable,block_0
'''
with open('/tmp/sample_data/sample_meters.csv', 'w') as f:
    f.write(sample_meters.strip())
with open('/tmp/sample_data/sample_households.csv', 'w') as f:
    f.write(sample_households.strip())

# Upload to HDFS /raw
!hdfs dfs -put -f /tmp/sample_data/sample_meters.csv /raw/
!hdfs dfs -put -f /tmp/sample_data/sample_households.csv /raw/
!hdfs dfs -ls /raw


## 6. Initializing PySpark Session Connecting to HDFS
## 6. تشغيل جلسة PySpark وربطها مع نظام HDFS


In [ ]:
!pip install -q pyspark
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName('LondonSmartGridAnalytics_Lab1')
    .master('local[*]')
    .config('spark.driver.memory', '4g')
    .getOrCreate()
)
sc = spark.sparkContext
print('Spark Master:', sc.master)


## 7. The 7 Core Low-Level RDD Operations
## 7. العمليات السبع الأساسية للـ RDDs في المعمل العملي


In [ ]:
raw_meters = sc.textFile('hdfs://localhost:9000/raw/sample_meters.csv')
raw_households = sc.textFile('hdfs://localhost:9000/raw/sample_households.csv')

# Op 1: filter - discard header, Null, negatives, malformed lines
def is_valid_line(l):
    if not l or 'LCLid' in l:
        return False
    parts = l.split(',')
    if len(parts) != 3:
        return False
    v = parts[2].strip()
    if v in ('Null', 'NULL', 'null', ''):
        return False
    try:
        return float(v) >= 0.0
    except ValueError:
        return False

valid_meters = raw_meters.filter(is_valid_line)
print('1. [filter] Valid records retained:', valid_meters.count())

# Op 2: map - parse to (LCLid, float(kwh))
pairs = valid_meters.map(lambda l: (l.split(',')[0].strip(), float(l.split(',')[2].strip())))
print('2. [map] Sample mapped pairs:', pairs.take(3))

# Op 3: flatMap - tokenize metadata
tokens = raw_households.filter(lambda l: 'LCLid' not in l).flatMap(
    lambda l: [t.strip() for t in l.split(',')[1:] if t.strip()]
)
print('3. [flatMap] Extracted metadata tokens:', tokens.take(5))

# Op 4: reduce - find global peak reading
peak = pairs.reduce(lambda a, b: a if a[1] >= b[1] else b)
print('4. [reduce] Global peak reading:', peak)

# Op 5: groupByKey - group readings per household
grouped = pairs.groupByKey().mapValues(list)
print('5. [groupByKey] Sample grouped readings:', grouped.take(1))

# Op 6: reduceByKey - sum energy per household
totals = pairs.reduceByKey(lambda a, b: a + b)
print('6. [reduceByKey] Totals per household:', totals.collect())

# Op 7: join - correlate with household tariffs (Std vs ToU)
tariffs = raw_households.filter(lambda l: 'LCLid' not in l).map(
    lambda l: (l.split(',')[0].strip(), l.split(',')[1].strip())
)
joined = totals.join(tariffs)
tariff_summary = joined.map(lambda x: (x[1][1], (x[1][0], 1))).reduceByKey(
    lambda a, b: (a[0] + b[0], a[1] + b[1])
).collect()

print('\n7. [join] Final Tariff Consumption Comparison:')
for t, (tot, cnt) in tariff_summary:
    print(f'{t}: {tot:.3f} kWh across {cnt} households (Avg: {tot/cnt:.3f} kWh)')


## 8. Driver Memory Protection & Checkpoint Verification
## 8. التحقق من حماية الذاكرة واكتمال المتطلبات

- **Constitutional Principle IV Verified**: Only 2 summary aggregate records collected to Driver.
- **Spec 01 Lab Syllabus 100% Satisfied**: HDFS setup, `/raw` ingestion, and all 7 RDD operations verified.
